# WellConverge health data — SageMaker Studio

Runs **inside SageMaker Unified Studio JupyterLab**. Unlike `01-explore-medical-data.ipynb`,
this notebook is self-contained — it does not import `wellconverge_tools`, so you can upload just
this file to a Studio space and run it.

It reads the Parquet files two ways:
1. **directly from S3** with pandas (fast, no query cost, good for ML)
2. **through the Glue Data Catalog** with Athena (SQL, partition pruning, what the catalog is for)

All data is **synthetic** — generated by `wc-tools`, no real patient information.

In [ ]:
BUCKET = "wellconverge-datalake-273505519511-ap-south-1"
PREFIX = "raw"
DATABASE = "wellconverge_health"
REGION = "ap-south-1"

# Use the project's own Athena workgroup — it already has a results location configured.
# Check the exact name with: aws athena list-work-groups --region ap-south-1
WORKGROUP = "sagemaker-studio-workgroup-aj7158g77gsfep"

import boto3, pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

session = boto3.Session(region_name=REGION)
print("running as:", session.client("sts").get_caller_identity()["Arn"])

## 1. What's in the lake

If `pd.read_parquet` on an `s3://` path fails with a missing-filesystem error, run
`%pip install s3fs` and restart the kernel.

In [ ]:
objects = session.client("s3").list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)["Contents"]
pd.DataFrame(
    [{"key": o["Key"], "MB": round(o["Size"] / 1e6, 2)} for o in objects]
).sort_values("key")

In [ ]:
# Hive-partitioned directories: pandas/pyarrow recovers `year` from the path automatically.
patients = pd.read_parquet(f"s3://{BUCKET}/{PREFIX}/patients")
encounters = pd.read_parquet(f"s3://{BUCKET}/{PREFIX}/encounters")
observations = pd.read_parquet(f"s3://{BUCKET}/{PREFIX}/observations")

for name, frame in [("patients", patients), ("encounters", encounters), ("observations", observations)]:
    print(f"{name:<14} {frame.shape[0]:>7,} rows x {frame.shape[1]:>2} cols")

encounters.head(3)

## 2. Read only one partition

The reason the data is partitioned: touch 2025 without reading 2023, 2024 or 2026.

In [ ]:
obs_2025 = pd.read_parquet(f"s3://{BUCKET}/{PREFIX}/observations/year=2025")
print(f"{len(obs_2025):,} observations in 2025 (vs {len(observations):,} total)")
obs_2025.groupby("observation_name")["value_num"].describe().round(2)

## 3. Same data through the Glue Data Catalog

This is the path that matters for the catalog: Athena resolves `wellconverge_health.encounters`
from Glue, and SageMaker Data Wrangler / Feature Store read the same metadata.

In [ ]:
import time


def athena(sql: str, timeout_s: float = 120.0) -> pd.DataFrame:
    """Run a query and return the results. Uses the workgroup's configured output location."""
    client = session.client("athena")
    query_id = client.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": DATABASE},
        WorkGroup=WORKGROUP,
    )["QueryExecutionId"]

    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        status = client.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]["Status"]
        if status["State"] in ("SUCCEEDED", "FAILED", "CANCELLED"):
            break
        time.sleep(1.5)
    else:
        raise TimeoutError(f"query {query_id} timed out")

    if status["State"] != "SUCCEEDED":
        raise RuntimeError(status.get("StateChangeReason", status["State"]))

    rows = client.get_query_results(QueryExecutionId=query_id)["ResultSet"]["Rows"]
    header = [c.get("VarCharValue") for c in rows[0]["Data"]]
    return pd.DataFrame(
        [[c.get("VarCharValue") for c in r["Data"]] for r in rows[1:]], columns=header
    )


athena("SHOW TABLES")

In [ ]:
athena("""
    SELECT primary_diagnosis_desc,
           count(*) AS encounters,
           round(avg(CASE WHEN readmitted_30d THEN 1.0 ELSE 0.0 END), 3) AS readmit_rate,
           round(approx_percentile(length_of_stay_days, 0.5), 2) AS median_los
    FROM encounters
    GROUP BY primary_diagnosis_desc
    ORDER BY readmit_rate DESC
""")

In [ ]:
# `WHERE year = ...` prunes partitions — Athena bills only the bytes it actually scans.
athena("""
    SELECT year, count(*) AS observations, round(avg(value_num), 2) AS avg_value
    FROM observations
    WHERE year = 2025 AND loinc_code = '4548-4'
    GROUP BY year
""")

## 4. Feature table + baseline model

In [ ]:
abnormal = (
    observations.assign(is_abnormal=observations["abnormal_flag"].ne("N").astype(int))
    .groupby("encounter_id", as_index=False)["is_abnormal"].sum()
    .rename(columns={"is_abnormal": "abnormal_observations"})
)

features = (
    encounters
    .merge(patients[["patient_id", "age_years", "sex", "bmi", "smoker",
                     "chronic_conditions", "insurance_type"]], on="patient_id", how="left")
    .merge(abnormal, on="encounter_id", how="left")
)
features["abnormal_observations"] = features["abnormal_observations"].fillna(0).astype(int)
print(f"{len(features):,} encounters x {features.shape[1]} columns")
features.head(3)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = ["age_years", "bmi", "chronic_conditions", "length_of_stay_days",
           "total_charge_usd", "abnormal_observations"]
CATEGORICAL = ["sex", "insurance_type", "encounter_class", "department",
               "primary_diagnosis_code", "smoker"]

X, y = features[NUMERIC + CATEGORICAL], features["readmitted_30d"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), CATEGORICAL),
    ])),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42)),
]).fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]
print(f"ROC-AUC {roc_auc_score(y_test, scores):.3f} | "
      f"PR-AUC {average_precision_score(y_test, scores):.3f} | "
      f"positives {y.mean():.1%}")

In [ ]:
coefficients = pd.DataFrame({
    "feature": model.named_steps["prep"].get_feature_names_out(),
    "coefficient": model.named_steps["clf"].coef_[0],
})
coefficients.reindex(coefficients["coefficient"].abs().sort_values(ascending=False).index).head(15)

---
**Before you close:** stop the JupyterLab space from the Unified Studio UI. The space bills per
hour while running; the S3 data and the Glue catalog cost effectively nothing.